In [1]:
# 1. 라이브러리 불러오기 및 실습 데이터 생성
# 실습을 위해 '고객 정보'를 담은 데이터프레임과 '주문 정보'를 담은 데이터프레임을 각각 생성합니다.
# (일부러 양쪽에 공통으로 존재하지 않는 고객ID를 섞어 두었습니다.)

import pandas as pd

# 1. 고객 데이터 (df_user) - ID 1, 2, 3, 5 존재
user_data = {
    '고객ID': [1, 2, 3, 5],
    '이름': ['홍길동', '김철수', '이영희', '박민준'],
    '가입지역': ['서울', '부산', '인천', '대전']
}
df_user = pd.DataFrame(user_data)

# 2. 주문 데이터 (df_order) - ID 1, 2, 3, 4 존재 (4번은 회원정보가 없는 비회원)
order_data = {
    '주문ID': [101, 102, 103, 104],
    '고객ID': [1, 2, 1, 4], # 1번 고객은 2번 주문함
    '상품명': ['노트북', '스마트폰', '마우스', '태블릿'],
    '수량': [1, 1, 2, 1]
}
df_order = pd.DataFrame(order_data)

print("--- [df_user] 고객 정보 ---")
print(df_user)
print("\n--- [df_order] 주문 정보 ---")
print(df_order)

--- [df_user] 고객 정보 ---
   고객ID   이름 가입지역
0     1  홍길동   서울
1     2  김철수   부산
2     3  이영희   인천
3     5  박민준   대전

--- [df_order] 주문 정보 ---
   주문ID  고객ID   상품명  수량
0   101     1   노트북   1
1   102     2  스마트폰   1
2   103     1   마우스   2
3   104     4   태블릿   1


In [2]:
# 2. pd.merge() 3대 핵심 조인 실습
# ① 내부 조인 (Inner Join) - 기본값
# •	how='inner' (생략 가능)

# •	기준: 두 데이터프레임 양쪽에 모두 존재하는 Key(고객ID) 값만 매칭하여 합칩니다.

# •	회원 정보도 있고 주문 기록도 있는 데이터만 남습니다. (고객ID 1, 2만 해당)

# on='고객ID'는 병합의 기준이 되는 공통 컬럼입니다.
df_inner = pd.merge(df_user, df_order, on='고객ID', how='inner')

print("--- [Inner Join] 양쪽 모두 존재하는 데이터만 합치기 ---")
print(df_inner)

--- [Inner Join] 양쪽 모두 존재하는 데이터만 합치기 ---
   고객ID   이름 가입지역  주문ID   상품명  수량
0     1  홍길동   서울   101   노트북   1
1     1  홍길동   서울   103   마우스   2
2     2  김철수   부산   102  스마트폰   1


In [3]:
# ② 왼쪽 조인 (Left Join)
# •	how='left'

# •	기준: 왼쪽 데이터프레임(df_user)의 모든 행을 유지한 채, 오른쪽 데이터프레임(df_order)의 매칭되는 데이터를 가져옵니다.

# •	특징: 주문을 한 번도 하지 않은 가입자(이영희, 박민준)도 데이터에 남게 되며, 이들의 주문 정보는 결측치(NaN)로 채워집니다. 반면 회원 정보가 없는 4번 주문 데이터는 사라집니다.

df_left = pd.merge(df_user, df_order, on='고객ID', how='left')

print("--- [Left Join] 왼쪽(고객) 기준, 주문이 없어도 고객 정보 유지 ---")
print(df_left)

--- [Left Join] 왼쪽(고객) 기준, 주문이 없어도 고객 정보 유지 ---
   고객ID   이름 가입지역   주문ID   상품명   수량
0     1  홍길동   서울  101.0   노트북  1.0
1     1  홍길동   서울  103.0   마우스  2.0
2     2  김철수   부산  102.0  스마트폰  1.0
3     3  이영희   인천    NaN   NaN  NaN
4     5  박민준   대전    NaN   NaN  NaN


In [4]:
# ③ 완전 외부 조인 (Outer Join)
# •	how='outer'

# •	기준: 양쪽 데이터프레임 중 어느 한 곳에라도 존재하는 모든 Key를 기준으로 데이터를 합칩니다.

# •	특징: 주문 안 한 회원(3, 5번) 정보도 유지되고, 회원 정보가 없는 비회원 주문(4번) 데이터도 유실 없이 모두 보존됩니다. 정보가 없는 빈칸은 전부 NaN 처리됩니다.

df_outer = pd.merge(df_user, df_order, on='고객ID', how='outer')

print("--- [Outer Join] 양쪽 데이터 유실 없이 모두 합치기 ---")
print(df_outer)

--- [Outer Join] 양쪽 데이터 유실 없이 모두 합치기 ---
   고객ID   이름 가입지역   주문ID   상품명   수량
0     1  홍길동   서울  101.0   노트북  1.0
1     1  홍길동   서울  103.0   마우스  2.0
2     2  김철수   부산  102.0  스마트폰  1.0
3     3  이영희   인천    NaN   NaN  NaN
4     4  NaN  NaN  104.0   태블릿  1.0
5     5  박민준   대전    NaN   NaN  NaN


In [5]:
# ※ 실무 트러블슈팅: 두 데이터프레임의 Key 컬럼명이 다를 때
# 실무에서는 합치려는 기준 컬럼의 이름이 서로 다르게 적혀있는 경우가 정말 많습니다. (예: 한쪽은 고객ID, 다른 쪽은 회원번호) 이럴 때는 on 대신 left_on과 right_on을 명시해야 합니다.

# 실습을 위해 컬럼명이 다른 데이터프레임 임의 변형
df_user_custom = df_user.rename(columns={'고객ID': '회원번호'})

# left_on에는 왼쪽 df의 컬럼명을, right_on에는 오른쪽 df의 컬럼명을 지정
df_custom_merge = pd.merge(df_user_custom, df_order, left_on='회원번호', right_on='고객ID', how='inner')

print("--- [응용] 컬럼명이 서로 다를 때 병합 결과 ---")
print(df_custom_merge)

--- [응용] 컬럼명이 서로 다를 때 병합 결과 ---
   회원번호   이름 가입지역  주문ID  고객ID   상품명  수량
0     1  홍길동   서울   101     1   노트북   1
1     1  홍길동   서울   103     1   마우스   2
2     2  김철수   부산   102     2  스마트폰   1
